# Task 2 — Clustering

03_clustering.py — Task 2: are there meaningful client groups?

Clustering is fit on the TRAIN period only
(same temporal split as 02_classification.py) so that comparing cluster
subscription rates on the holdout at the end is a genuine test, not
something the clustering could have seen.

y, duration, and any model prediction are excluded from the clustering
features throughout.

In [55]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42

## 1. Load + same temporal split as Task 1 (no holdout leakage into Task 2)

In [56]:
df = pd.read_csv("data/bank-additional-full.csv", sep=";")
df["y"] = (df["y"] == "yes").astype(int)
df["was_contacted_before"] = (df["pdays"] != 999).astype(int)

split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
holdout_df = df.iloc[split_idx:].copy()

## 2. Define the clustering unit

In [57]:
# DECISION: the dataset has no client ID, and the same person may appear
# multiple times across contacts with different attribute snapshots. We
# therefore cluster CONTACT RECORDS, not "unique clients" — we cannot
# support the stronger claim. This is stated explicitly rather than
# assumed, per the assignment.

## 3. Feature selection — client attributes + past-campaign info only

In [58]:
# DECISION: economic indicators are EXCLUDED from the main clustering run.
# They vary mostly by date, not by client, so including them risks
# clustering "which month this contact happened" rather than "what kind
# of client this is" — exactly the caveat the assignment flags. If you
# want to test this, rerun with them added and compare cluster profiles;
# don't silently decide either way without checking.
numeric_cols = ["age", "campaign", "pdays", "previous", "was_contacted_before"]
categorical_cols = [
    "job", "marital", "education", "default", "housing", "loan",
    "contact", "month", "day_of_week", "poutcome",
]
feature_cols = numeric_cols + categorical_cols

X_train = train_df[feature_cols]
X_holdout = holdout_df[feature_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         categorical_cols),
    ]
)
X_train_enc = preprocess.fit_transform(X_train)
X_holdout_enc = preprocess.transform(X_holdout)
# NOTE: with one-hot categoricals dominating the column count, Euclidean
# distance here is really closer to "how many categorical fields match"
# plus a modest numeric contribution — worth stating explicitly, since it
# shapes what "similar" means for every clustering method below.

## 4. Compare clustering approaches

In [59]:
print("K-MEANS: silhouette vs k (on full train set)")
# DECISION: a documented, reproducible sample for silhouette scoring,
# since silhouette is O(n^2) — too slow at 33k rows for repeated scoring.
rng = np.random.default_rng(RANDOM_STATE)
sample_idx = rng.choice(len(X_train_enc), size=4000, replace=False)
X_sample = X_train_enc[sample_idx]

kmeans_scores = {}
for k in [3, 4, 5, 6]:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels_full = km.fit_predict(X_train_enc)
    sil = silhouette_score(X_sample, labels_full[sample_idx])
    kmeans_scores[k] = sil
    print(f"  k={k}: silhouette (n=4000 sample) = {sil:.4f}")

best_k = max(kmeans_scores, key=kmeans_scores.get)
print(f"\nSelected k={best_k} for K-means (highest sampled silhouette)")
# A low absolute silhouette (well under ~0.3) across all k would itself be
# a real result — it would mean the data doesn't separate into tight,
# well-defined groups under this representation. Report whatever you see.

kmeans_final = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
train_labels = kmeans_final.fit_predict(X_train_enc)

print("\n")
print("AGGLOMERATIVE (Ward) on the SAME reproducible 4000-row sample")
for k in [3, 4, 5, 6]:
    agg = AgglomerativeClustering(n_clusters=k, linkage="ward")
    agg_labels = agg.fit_predict(X_sample)
    sil = silhouette_score(X_sample, agg_labels)
    print(f"  k={k}: silhouette = {sil:.4f}")
    if k == best_k:
        ari = adjusted_rand_score(train_labels[sample_idx], agg_labels)
        print(f"    agreement with K-means labels on this sample "
              f"(Adjusted Rand Index) = {ari:.4f}")
# A low ARI here means the two methods are finding genuinely different
# structure, not just relabelling the same groups — note it either way.

K-MEANS: silhouette vs k (on full train set)
  k=3: silhouette (n=4000 sample) = 0.3225
  k=4: silhouette (n=4000 sample) = 0.1274
  k=5: silhouette (n=4000 sample) = 0.1311
  k=6: silhouette (n=4000 sample) = 0.1146

Selected k=3 for K-means (highest sampled silhouette)


AGGLOMERATIVE (Ward) on the SAME reproducible 4000-row sample
  k=3: silhouette = 0.3225
    agreement with K-means labels on this sample (Adjusted Rand Index) = 1.0000
  k=4: silhouette = 0.2570
  k=5: silhouette = 0.1136
  k=6: silhouette = 0.0924


## 5. Validate the chosen K-means solution

In [60]:
print(f"CLUSTER SIZES (k={best_k}, full train set, n={len(train_df)})")
sizes = pd.Series(train_labels).value_counts().sort_index()
print(sizes)
print(f"Smallest cluster is {sizes.min() / len(train_df):.2%} of the data "
      f"— flag if any cluster is tiny (<1-2%) or near-duplicate of another.")

print("\n")
print("STABILITY ACROSS SEEDS")
alt_labels = KMeans(
    n_clusters=best_k, random_state=RANDOM_STATE + 1, n_init=10
).fit_predict(X_train_enc)
stability_ari = adjusted_rand_score(train_labels, alt_labels)
print(f"ARI between seed={RANDOM_STATE} and seed={RANDOM_STATE + 1} "
      f"solutions: {stability_ari:.4f}  (closer to 1 = more stable)")

CLUSTER SIZES (k=3, full train set, n=32950)
0     2101
1    30675
2      174
Name: count, dtype: int64
Smallest cluster is 0.53% of the data — flag if any cluster is tiny (<1-2%) or near-duplicate of another.


STABILITY ACROSS SEEDS
ARI between seed=42 and seed=43 solutions: 0.0100  (closer to 1 = more stable)


## 6. Cluster profiles

In [61]:
profile_df = train_df[feature_cols].copy()
profile_df["cluster"] = train_labels

print("CLUSTER PROFILES — numeric means")
print(profile_df.groupby("cluster")[numeric_cols].mean().round(2))

print("\n")
print("CLUSTER PROFILES — most common category per field")
for col in categorical_cols:
    modes = profile_df.groupby("cluster")[col].agg(
        lambda s: s.value_counts().idxmax()
    )
    print(f"  {col:12s}: {modes.tolist()}")

CLUSTER PROFILES — numeric means
           age  campaign   pdays  previous  was_contacted_before
cluster                                                         
0        39.95      1.88  999.00      1.05                   0.0
1        40.03      2.73  999.00      0.00                   0.0
2        41.17      1.94    5.29      1.14                   1.0


CLUSTER PROFILES — most common category per field
  job         : ['blue-collar', 'admin.', 'admin.']
  marital     : ['married', 'married', 'married']
  education   : ['university.degree', 'university.degree', 'university.degree']
  default     : ['no', 'no', 'no']
  housing     : ['yes', 'yes', 'yes']
  loan        : ['no', 'no', 'no']
  contact     : ['cellular', 'cellular', 'cellular']
  month       : ['may', 'may', 'may']
  day_of_week : ['fri', 'thu', 'thu']
  poutcome    : ['failure', 'nonexistent', 'success']


## 7. PCA visualisation (inspection only — not proof of separation)

In [62]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X_sample)
plt.figure(figsize=(7, 6))
scatter = plt.scatter(
    coords[:, 0], coords[:, 1], c=train_labels[sample_idx],
    cmap="tab10", s=8, alpha=0.6,
)
plt.title(f"K-means clusters (k={best_k}), 2D PCA projection "
          f"(explains {pca.explained_variance_ratio_.sum():.1%} of variance)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(scatter, label="cluster")
plt.tight_layout()
plt.savefig("outputs/cluster_pca.png", dpi=120)
print(f"\nSaved cluster_pca.png — PC1+PC2 explain only "
      f"{pca.explained_variance_ratio_.sum():.1%} of variance, so this 2D "
      f"view is an inspection aid, NOT proof the clusters are well "
      f"separated in the original feature space.")


Saved cluster_pca.png — PC1+PC2 explain only 31.7% of variance, so this 2D view is an inspection aid, NOT proof the clusters are well separated in the original feature space.


## 8. Connect to Task 1 — does segmentation add anything?

In [63]:
holdout_labels = kmeans_final.predict(X_holdout_enc)
holdout_eval = holdout_df.copy()
holdout_eval["cluster"] = holdout_labels

print("\n")
print("HOLDOUT SUBSCRIPTION RATE BY CLUSTER (clusters fixed from TRAIN only)")
rate_by_cluster = holdout_eval.groupby("cluster")["y"].agg(["mean", "count"])
rate_by_cluster.columns = ["subscription_rate", "n_holdout_rows"]
overall_rate = holdout_eval["y"].mean()
rate_by_cluster["vs_overall"] = (
    rate_by_cluster["subscription_rate"] - overall_rate
).round(4)
print(rate_by_cluster.round(4))
print(f"\nOverall holdout subscription rate: {overall_rate:.4f}")
print(
    "Read this yourself: if subscription rate is roughly flat across "
    "clusters, segmentation adds little for THIS campaign decision, and "
    "that is a legitimate, reportable conclusion — don't force a 'the "
    "clusters matter' narrative that the numbers don't support. "
    "Correlation here is not a tested intervention either way."
)



HOLDOUT SUBSCRIPTION RATE BY CLUSTER (clusters fixed from TRAIN only)
         subscription_rate  n_holdout_rows  vs_overall
cluster                                               
0                   0.2041            2009     -0.1042
1                   0.2514            4888     -0.0569
2                   0.6719            1341      0.3636

Overall holdout subscription rate: 0.3083
Read this yourself: if subscription rate is roughly flat across clusters, segmentation adds little for THIS campaign decision, and that is a legitimate, reportable conclusion — don't force a 'the clusters matter' narrative that the numbers don't support. Correlation here is not a tested intervention either way.
